In [0]:
# =============================================================================
# Ingestão parametrizada — wrapper fino do motor genérico (motores.py).
# Parâmetros (widgets/job params): taxi_type, catalogo, ano, meses, origem
#   origem=raw  -> copia os parquets do volume raw para o landing (motor)
#   origem=url  -> baixa da fonte oficial (url_template do contrato) direto
#                  para o landing — use para ingerir datas novas
# O dataset é definido pelo contrato contracts/{taxi_type}_trips.yml.
# =============================================================================
dbutils.widgets.text("taxi_type", "yellow", "taxi_type (yellow|green)")
dbutils.widgets.text("catalogo", "case_ifood", "catalogo UC")
dbutils.widgets.text("ano", "2023", "ano")
dbutils.widgets.text("meses", "01,02,03,04,05", "meses (csv)")
dbutils.widgets.dropdown("origem", "raw", ["raw", "url"], "origem dos parquets")

taxi_type = dbutils.widgets.get("taxi_type")
catalogo = dbutils.widgets.get("catalogo")
ano = dbutils.widgets.get("ano")
meses = [m.strip().zfill(2) for m in dbutils.widgets.get("meses").split(",") if m.strip()]
origem = dbutils.widgets.get("origem")

import sys
sys.path.insert(0, "/Workspace/Repos/alexvalsechibosque@gmail.com/case-ifood/00_config")
import motores

contrato = motores.carregar_contrato(f"{taxi_type}_trips")
print(f"Contrato carregado: {contrato['dataset']} (grain: {contrato['descricao'].strip().splitlines()[0]})")
print(f"Destinos: bronze={contrato['destinos']['bronze']} | silver={contrato['destinos']['silver']}")

In [0]:
# Executa o motor genérico conforme a origem escolhida:
# - url: baixa da fonte oficial (url_template do contrato) para o landing
# - raw: copia do volume raw para o landing (filtro por ano/meses)
if origem == "url":
    resultados = motores.baixar_da_fonte(contrato, spark=spark, ano=ano, meses=meses, catalogo=catalogo)
else:
    filtro = [f"{contrato['taxi_type']}_tripdata_{ano}-{m}.parquet" for m in meses]
    resultados = motores.executar_ingestao(
        contrato,
        spark=spark,
        catalogo=catalogo,
        filtro_arquivos=filtro,
    )

import pandas as pd
display(pd.DataFrame(resultados))